# Persian Book Cover Lens

## Persian Book Title Recognition from Cover Images using Vision-Language Models

Using a Vision-Language Model (VLM) to extract Persian book titles from cover images, then fine-tuning the model to improve accuracy.

This notebook is designed for **Google Colab**. It implements:

1. Dataset loading from Hugging Face  
2. Train/test subset selection  
3. Zero-shot VLM evaluation  
4. Persian text normalization  
5. Exact Match, Levenshtein Similarity, and Word-level F1  
6. LoRA/QLoRA fine-tuning  
7. Post-fine-tuning evaluation  
8. Metric comparison chart and CSV outputs  

> Default mode is `FAST_DEV_RUN=True`, so the notebook can be tested quickly. For the final course run, switch it to `False`.

## 0. Runtime Notes

Recommended Colab runtime:

- **Runtime type:** GPU
- **GPU:** T4, L4, A100, or similar
- **Python:** Colab default Python
- **Model:** `Qwen/Qwen2.5-VL-3B-Instruct`

If no GPU is available, the notebook still runs the dataset and metric sections, but real VLM inference and fine-tuning are skipped. This is intentional to avoid fake results and memory crashes.

In [ ]:
# Check GPU
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
else:
    print("No GPU detected. Dataset and metric cells will run; VLM inference/fine-tuning will be skipped.")

## 1. Install Dependencies

Qwen2.5-VL support may require a recent `transformers` version. Installing from the Hugging Face GitHub repository avoids the common `KeyError: 'qwen2_5_vl'` issue.

In [ ]:
%%capture
!pip install -U -q git+https://github.com/huggingface/transformers
!pip install -U -q accelerate datasets peft trl bitsandbytes qwen-vl-utils rapidfuzz pandas numpy matplotlib pillow tqdm

## 2. Imports and Reproducibility

In [ ]:
import os
import re
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from PIL import Image, ImageEnhance
from datasets import load_dataset
from rapidfuzz.distance import Levenshtein

import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 3. Project Configuration

For a quick smoke test, keep `FAST_DEV_RUN=True`.

For the final assignment run:

```python
FAST_DEV_RUN = False
TRAIN_SUBSET_SIZE = 1000
TEST_SUBSET_SIZE = 200
```

In [ ]:
FAST_DEV_RUN = True

DATASET_ID = "shenasa/bookroom-persian-book-covers-and-titles"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

TRAIN_SUBSET_SIZE = 12 if FAST_DEV_RUN else 1000
TEST_SUBSET_SIZE = 4 if FAST_DEV_RUN else 200

# Small images reduce GPU memory pressure.
# Qwen visual tokens are based on 28x28 patches.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = (384 if FAST_DEV_RUN else 768) * 28 * 28

PROMPT = (
    "Extract only the main Persian book title from this cover image. "
    "Return only the title text. Do not include author, publisher, price, edition, or explanation."
)

OUTPUT_DIR = Path("/content/persian_book_title_vlm_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("FAST_DEV_RUN:", FAST_DEV_RUN)
print("Train subset size:", TRAIN_SUBSET_SIZE)
print("Test subset size:", TEST_SUBSET_SIZE)
print("Output directory:", OUTPUT_DIR)

## 4. Load the Dataset

The dataset contains Persian book-cover images and their title text. We use the `train` split for fine-tuning and the first samples from the `test` split for evaluation.

In [ ]:
dataset = load_dataset(DATASET_ID)

print(dataset)
print("Train columns:", dataset["train"].column_names)
print("Test columns:", dataset["test"].column_names)

IMAGE_COLUMN = "image"
TEXT_COLUMN = "text"

train_ds = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SUBSET_SIZE))
test_ds = dataset["test"].select(range(TEST_SUBSET_SIZE))

print("Selected train samples:", len(train_ds))
print("Selected test samples:", len(test_ds))

## 5. Inspect a Few Samples

This is a simple sanity check: image and ground-truth Persian title should match visually.

In [ ]:
def show_samples(ds, n=3):
    n = min(n, len(ds))
    plt.figure(figsize=(4 * n, 5))
    for i in range(n):
        sample = ds[i]
        img = sample[IMAGE_COLUMN].convert("RGB")
        title = sample[TEXT_COLUMN]
        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

show_samples(test_ds, n=min(3, len(test_ds)))

## 6. Persian Text Normalization

Exact matching is very strict. Minor differences such as Arabic/Persian characters, extra spaces, or punctuation can unfairly reduce scores. We therefore report metrics after a light normalization step.

In [ ]:
PERSIAN_DIACRITICS = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")

def normalize_persian_text(text: str) -> str:
    """Normalize Persian/Arabic text for fair title comparison."""
    if text is None:
        return ""
    text = str(text)
    text = text.strip()
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = text.replace("ۀ", "ه").replace("ة", "ه")
    text = text.replace("\u200c", " ")  # zero-width non-joiner
    text = PERSIAN_DIACRITICS.sub("", text)
    text = re.sub(r"[ـ]", "", text)
    text = re.sub(r"[\"'`“”‘’]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_prediction(text: str) -> str:
    """Remove common extra phrases and keep a compact title string."""
    text = normalize_persian_text(text)
    prefixes = [
        "عنوان کتاب:",
        "عنوان:",
        "نام کتاب:",
        "The title is",
        "Book title:",
    ]
    for p in prefixes:
        text = text.replace(p, "")
    text = text.strip(" .:：-")
    return normalize_persian_text(text)

# Quick metric sanity examples
examples = [
    ("روزنه ای به چشمه های نور", "روزنه‌ای به چشمه‌های نور"),
    ("ساواک و مرجعیت", "ساواک و مرجعیت"),
    ("پسر خدا", "پسر خدا "),
]
for a, b in examples:
    print(a, "=>", normalize_persian_text(a))
    print(b, "=>", normalize_persian_text(b))
    print()

## 7. Evaluation Metrics

We use three metrics:

- **Exact Match:** strict title-level correctness after normalization.
- **Levenshtein Similarity:** character-level similarity; useful when the prediction is close but not exact.
- **Word-level F1:** evaluates overlap at word level.

In [ ]:
def exact_match(reference: str, prediction: str) -> float:
    return float(normalize_persian_text(reference) == clean_prediction(prediction))

def levenshtein_similarity(reference: str, prediction: str) -> float:
    ref = normalize_persian_text(reference)
    pred = clean_prediction(prediction)
    if len(ref) == 0 and len(pred) == 0:
        return 1.0
    max_len = max(len(ref), len(pred), 1)
    dist = Levenshtein.distance(ref, pred)
    return 1.0 - (dist / max_len)

def word_level_f1(reference: str, prediction: str) -> float:
    ref_tokens = normalize_persian_text(reference).split()
    pred_tokens = clean_prediction(prediction).split()
    if not ref_tokens and not pred_tokens:
        return 1.0
    if not ref_tokens or not pred_tokens:
        return 0.0

    ref_counts = {}
    for t in ref_tokens:
        ref_counts[t] = ref_counts.get(t, 0) + 1

    overlap = 0
    for t in pred_tokens:
        if ref_counts.get(t, 0) > 0:
            overlap += 1
            ref_counts[t] -= 1

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def evaluate_predictions(df: pd.DataFrame) -> dict:
    em = [exact_match(r, p) for r, p in zip(df["reference"], df["prediction"])]
    lev = [levenshtein_similarity(r, p) for r, p in zip(df["reference"], df["prediction"])]
    f1 = [word_level_f1(r, p) for r, p in zip(df["reference"], df["prediction"])]

    return {
        "exact_match": float(np.mean(em)) if em else 0.0,
        "levenshtein_similarity": float(np.mean(lev)) if lev else 0.0,
        "word_f1": float(np.mean(f1)) if f1 else 0.0,
        "n_samples": len(df),
    }

# Unit test
test_eval = pd.DataFrame({
    "reference": ["پسر خدا", "نمک در چشمانم می ریزم"],
    "prediction": ["پسر خدا", "نمک در چشمانم می ریزد"],
})
evaluate_predictions(test_eval)

## 8. Load the VLM

We use `Qwen/Qwen2.5-VL-3B-Instruct` because it is a relatively small instruction-tuned VLM from the Qwen2.5-VL family and is suitable for OCR-like image-to-text tasks.

The model is loaded with 4-bit quantization on GPU to reduce memory usage.

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

def get_compute_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

def load_qwen25_vl(model_id=MODEL_ID):
    if not torch.cuda.is_available():
        print("No CUDA GPU. Skipping model loading.")
        return None, None

    compute_dtype = get_compute_dtype()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    processor = AutoProcessor.from_pretrained(
        model_id,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        trust_remote_code=True,
    )
    processor.tokenizer.padding_side = "right"

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=compute_dtype,
        device_map="auto",
        trust_remote_code=True,
    )

    model.eval()
    print("Model loaded:", model_id)
    return model, processor

model, processor = load_qwen25_vl()

## 9. Baseline Zero-Shot Inference

This section evaluates the base VLM before fine-tuning.

In [ ]:
def make_user_messages():
    return [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

@torch.inference_mode()
def predict_title(image: Image.Image, model, processor, max_new_tokens=32) -> str:
    if model is None or processor is None:
        return ""

    image = image.convert("RGB")
    messages = make_user_messages()
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

    generated_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]
    output = processor.batch_decode(
        generated_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return clean_prediction(output)

def run_inference(ds, model, processor, split_name="baseline"):
    rows = []
    for idx, sample in enumerate(tqdm(ds, desc=f"{split_name} inference")):
        reference = sample[TEXT_COLUMN]
        prediction = predict_title(sample[IMAGE_COLUMN], model, processor)
        rows.append({
            "idx": idx,
            "reference": reference,
            "prediction": prediction,
            "reference_norm": normalize_persian_text(reference),
            "prediction_norm": clean_prediction(prediction),
        })
    return pd.DataFrame(rows)

baseline_df = run_inference(test_ds, model, processor, split_name="baseline")
baseline_metrics = evaluate_predictions(baseline_df)

baseline_path = OUTPUT_DIR / "baseline_predictions.csv"
baseline_df.to_csv(baseline_path, index=False, encoding="utf-8-sig")

print("Baseline metrics:")
print(json.dumps(baseline_metrics, ensure_ascii=False, indent=2))
print("Saved:", baseline_path)
baseline_df.head()

## 10. Prepare Data for Fine-Tuning

Each training example is converted into a chat-style VLM instruction:

- **User:** image + extraction instruction  
- **Assistant:** ground-truth Persian title

In [ ]:
def build_prompt_text(processor):
    user_messages = make_user_messages()
    return processor.apply_chat_template(
        user_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def build_full_text(processor, title: str):
    messages = make_user_messages() + [
        {
            "role": "assistant",
            "content": normalize_persian_text(title),
        }
    ]
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

def simple_augment_image(image: Image.Image) -> Image.Image:
    """Light augmentation: small rotation and brightness jitter."""
    image = image.convert("RGB")
    if random.random() < 0.5:
        angle = random.uniform(-3, 3)
        image = image.rotate(angle, resample=Image.Resampling.BICUBIC, expand=False)
    if random.random() < 0.5:
        factor = random.uniform(0.85, 1.15)
        image = ImageEnhance.Brightness(image).enhance(factor)
    return image

ENABLE_AUGMENTATION = False  # Set True for the final experiment if desired.

class VLMTitleDataCollator:
    def __init__(self, processor, image_column, text_column, enable_augmentation=False):
        self.processor = processor
        self.image_column = image_column
        self.text_column = text_column
        self.enable_augmentation = enable_augmentation

    def __call__(self, features):
        images = []
        full_texts = []
        prompt_texts = []

        for feature in features:
            image = feature[self.image_column].convert("RGB")
            if self.enable_augmentation:
                image = simple_augment_image(image)

            title = feature[self.text_column]
            images.append(image)
            full_texts.append(build_full_text(self.processor, title))
            prompt_texts.append(build_prompt_text(self.processor))

        batch = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        prompt_batch = self.processor(
            text=prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        labels = batch["input_ids"].clone()
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)

        for i, prompt_len in enumerate(prompt_lengths):
            labels[i, : int(prompt_len.item())] = -100

        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        batch["labels"] = labels
        return batch

if processor is not None:
    collator = VLMTitleDataCollator(
        processor=processor,
        image_column=IMAGE_COLUMN,
        text_column=TEXT_COLUMN,
        enable_augmentation=ENABLE_AUGMENTATION,
    )
    small_batch = collator([train_ds[0]])
    print("Collator keys:", small_batch.keys())
    print("input_ids shape:", small_batch["input_ids"].shape)
else:
    print("Processor is not loaded because no GPU was detected.")

## 11. LoRA/QLoRA Fine-Tuning

We fine-tune only small LoRA adapter weights instead of the full model. This is more practical for Colab and is appropriate for a course-level VLM fine-tuning assignment.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer

def apply_lora(model):
    if model is None:
        return None

    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8 if FAST_DEV_RUN else 16,
        lora_alpha=16 if FAST_DEV_RUN else 32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

if model is None or processor is None:
    print("Skipping fine-tuning because no GPU/model is available.")
    finetuned_model = None
else:
    finetuned_model = apply_lora(model)

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "checkpoints"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=1,
        max_steps=2 if FAST_DEV_RUN else -1,
        logging_steps=1,
        save_strategy="no" if FAST_DEV_RUN else "epoch",
        fp16=(get_compute_dtype() == torch.float16),
        bf16=(get_compute_dtype() == torch.bfloat16),
        optim="paged_adamw_8bit",
        remove_unused_columns=False,
        report_to="none",
    )

    trainer = Trainer(
        model=finetuned_model,
        args=training_args,
        train_dataset=train_ds,
        data_collator=collator,
    )

    trainer.train()

    adapter_dir = OUTPUT_DIR / "qwen25vl_persian_title_lora"
    finetuned_model.save_pretrained(adapter_dir)
    processor.save_pretrained(adapter_dir)

    print("Saved LoRA adapter to:", adapter_dir)

## 12. Evaluation After Fine-Tuning

We evaluate the fine-tuned adapter on the **same test subset** used for the baseline.

In [ ]:
if finetuned_model is None or processor is None:
    print("Fine-tuned model is not available. Creating an empty post-fine-tuning result table.")
    finetuned_df = baseline_df.copy()
    finetuned_df["prediction"] = ""
    finetuned_df["prediction_norm"] = ""
else:
    finetuned_model.eval()
    finetuned_df = run_inference(test_ds, finetuned_model, processor, split_name="fine-tuned")

finetuned_metrics = evaluate_predictions(finetuned_df)

finetuned_path = OUTPUT_DIR / "finetuned_predictions.csv"
finetuned_df.to_csv(finetuned_path, index=False, encoding="utf-8-sig")

print("Fine-tuned metrics:")
print(json.dumps(finetuned_metrics, ensure_ascii=False, indent=2))
print("Saved:", finetuned_path)
finetuned_df.head()

## 13. Compare Baseline and Fine-Tuned Results

In [ ]:
comparison_df = pd.DataFrame([
    {"stage": "Base VLM", **baseline_metrics},
    {"stage": "Fine-tuned VLM", **finetuned_metrics},
])

comparison_path = OUTPUT_DIR / "metric_comparison.csv"
comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")

display(comparison_df)

plot_df = comparison_df.set_index("stage")[["exact_match", "levenshtein_similarity", "word_f1"]] * 100

ax = plot_df.plot(kind="bar", figsize=(9, 5))
ax.set_title("Persian Book Title Recognition: Before vs After Fine-Tuning")
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()

plot_path = OUTPUT_DIR / "metric_comparison.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved comparison table:", comparison_path)
print("Saved chart:", plot_path)

## 14. Error Analysis

This section shows the most difficult samples according to Levenshtein similarity.

In [ ]:
error_df = finetuned_df.copy()
error_df["exact_match"] = [
    exact_match(r, p) for r, p in zip(error_df["reference"], error_df["prediction"])
]
error_df["levenshtein_similarity"] = [
    levenshtein_similarity(r, p) for r, p in zip(error_df["reference"], error_df["prediction"])
]
error_df["word_f1"] = [
    word_level_f1(r, p) for r, p in zip(error_df["reference"], error_df["prediction"])
]

error_df = error_df.sort_values("levenshtein_similarity", ascending=True)

error_path = OUTPUT_DIR / "error_analysis.csv"
error_df.to_csv(error_path, index=False, encoding="utf-8-sig")

print("Saved error analysis:", error_path)
error_df.head(10)

## 15. Final Report Template

Use this summary in your README or course report after running the final experiment with `FAST_DEV_RUN=False`.

Replace the metric values with your actual final output table.

In [ ]:
def pct(x):
    return f"{100*x:.2f}%"

report = f"""
Final experiment summary

Dataset:
- Train subset: {TRAIN_SUBSET_SIZE} samples from the train split
- Test subset: {TEST_SUBSET_SIZE} samples from the test split

Model:
- Base VLM: {MODEL_ID}
- Fine-tuning method: LoRA / QLoRA supervised fine-tuning

Baseline:
- Exact Match: {pct(baseline_metrics['exact_match'])}
- Levenshtein Similarity: {pct(baseline_metrics['levenshtein_similarity'])}
- Word-level F1: {pct(baseline_metrics['word_f1'])}

After fine-tuning:
- Exact Match: {pct(finetuned_metrics['exact_match'])}
- Levenshtein Similarity: {pct(finetuned_metrics['levenshtein_similarity'])}
- Word-level F1: {pct(finetuned_metrics['word_f1'])}
"""
print(report)

## 16. How to Run the Final Version

1. Open this notebook in Google Colab.
2. Select **Runtime → Change runtime type → GPU**.
3. Run all cells once with `FAST_DEV_RUN=True`.
4. If the smoke test works, set:
   ```python
   FAST_DEV_RUN = False
   TRAIN_SUBSET_SIZE = 1000
   TEST_SUBSET_SIZE = 200
   ```
5. Run again.
6. Download the generated files from:
   ```text
   /content/persian_book_title_vlm_outputs
   ```
7. Upload the notebook, README, CSV results, and chart to GitHub.